# 1. Extraction des données

Ce notebook extrait les paragraphes des comptes rendus de l’Assemblée nationale à partir des fichiers XML (en utilisant la bibliothèque lxml) et retourne un csv exploitable dans la suite de l'analyse.

La dernière section (1.4) permet si souhaité de regrouper les interventions interrompues.

In [11]:
# TODO: plus tard, aviser récupération des points de contexte parents(cf tentative Matthias)
# TODO: est-ce que les séquences d'interruption peuvent être identifiées directement depuis xml car elles seraient dans un bloc <interExtraction> ?


## 1.1 définition fonctions d'extraction

In [12]:
import os
import glob
from lxml import etree
import pandas as pd

# ==================================================================
# FONCTIONS D'EXTRACTION DES DONNÉES
# ==================================================================


# ======== Fonction extraction infos depuis fichier XML =========
def extraire_paragraphes_lxml(fichier_xml: str) -> pd.DataFrame:
    """
    Extrait les paragraphes d'un fichier XML de compte rendu en utilisant lxml.
    """
    try:
        tree = etree.parse(fichier_xml)
        root = tree.getroot()
        ns = {"ns": "http://schemas.assemblee-nationale.fr/referentiel"}

        meta = {
            "uid": root.findtext("ns:uid", namespaces=ns),
            "SeanceRef": root.findtext("ns:seanceRef", namespaces=ns),  # pas partout
            "SessionRef": root.findtext("ns:sessionRef", namespaces=ns),  # pas partout
        }
        meta_tags = [
            "dateSeance",
            "dateSeanceJour",
            "numSeanceJour",
            "numSeance",
            "typeAssemblee",
            "legislature",
            "session",
            "nomFichierJo",
            "presidentSeance",
        ]
        for tag in meta_tags:
            meta[tag] = root.findtext(f".//ns:{tag}", namespaces=ns)

        rows = []

        for paragraphe in root.xpath(".//ns:paragraphe", namespaces=ns):
            # Naviguer vers le <point> parent
            # récupérer les infos
            point = paragraphe.getparent()
            while point is not None and point.tag != f"{{{ns['ns']}}}point":
                point = point.getparent()

            point_type = point.get("code_grammaire") if point is not None else None

            # anciennement : utilisait findtext(), mais ignore les sous-balises donc perte de texte
            # plutôt utiliser itertext() pour reconstruire le contenu complet
            texte_point = (
                point.find("ns:texte", namespaces=ns) if point is not None else None
            )
            point_title = (
                "".join(texte_point.itertext()).strip()
                if texte_point is not None
                else None
            )

            # # Plus pris pour l'instant (ie niveau du point, on a toujours niveau paragraphe plus bas):
            # point_id = point.get("id_syceron") if point is not None else None
            # point_valeur_ptsodj = point.get("valeur_ptsodj") if point is not None else None

            texte_elem = paragraphe.find("ns:texte", namespaces=ns)
            texte = (
                "".join(texte_elem.itertext()).strip()
                if texte_elem is not None
                else None
            )
            stime = texte_elem.get("stime") if texte_elem is not None else None

            # Récupérer les informations de l'orateur
            # ie celles présentes dans la balise <orateur>
            # et pas forcément dans les attributs du paragraphe
            orateur = paragraphe.find(".//ns:orateur", namespaces=ns)
            nom_orateur = (
                orateur.findtext("ns:nom", namespaces=ns)
                if orateur is not None
                else None
            )
            qualite_orateur = (
                orateur.findtext("ns:qualite", namespaces=ns)
                if orateur is not None
                else None
            )
            id_orateur = (
                orateur.findtext("ns:id", namespaces=ns)
                if orateur is not None
                else None
            )

            # toper désormais toutes les infos
            # garder apparent pour éventuels choix ou recodages des noms plutôt que des machins type `**meta`
            rows.append(
                {
                    # ========================
                    # Métadonnées de la séance
                    # ========================
                    "uid": meta["uid"],
                    "SeanceRef": meta["SeanceRef"],
                    "SessionRef": meta["SessionRef"],
                    "dateSeance": meta["dateSeance"],
                    "dateSeanceJour": meta["dateSeanceJour"],
                    "numSeanceJour": meta["numSeanceJour"],
                    "numSeance": meta["numSeance"],
                    "typeAssemblee": meta["typeAssemblee"],
                    "legislature": meta["legislature"],
                    "session": meta["session"],
                    "nomFichierJo": meta["nomFichierJo"],
                    "presidentSeance": meta["presidentSeance"],
                    # ========================
                    # Données du point parent (contexte)
                    # ========================
                    "point_titre": point_title,
                    "point_type": point_type,
                    # 'Sous_titre': '',  # not in this version, get back to original if needed
                    # 'Contexte_hierarchique': '',  # not in this version, get back to original if needed
                    # 'Section_courante': '',  # not in this version, get back to original if needed
                    # 'Sujet_point': '', # not in this version, get back to original if needed
                    # "point_valeur_ptsodj": point_valeur_ptsodj,
                    # "point_id": point_id,
                    # ========================
                    # données du paragraphe
                    # ========================
                    "valeur_ptsodj": paragraphe.get("valeur_ptsodj"),
                    "ordinal_prise": paragraphe.get("ordinal_prise"),
                    "ordre_absolu_seance": paragraphe.get("ordre_absolu_seance"),
                    "id_acteur": paragraphe.get("id_acteur"),
                    "id_mandat": paragraphe.get("id_mandat"),
                    "code_grammaire": paragraphe.get("code_grammaire"),
                    "code_style": paragraphe.get("code_style"),
                    "code_parole": paragraphe.get("code_parole"),
                    "id_syceron": paragraphe.get("id_syceron"),
                    "roledebat": paragraphe.get("roledebat"),
                    # ========================
                    # données orateur + texte
                    # ========================
                    "nom_orateur": nom_orateur,
                    "qualite_orateur": qualite_orateur,
                    "id_orateur": id_orateur,
                    "stime": stime,
                    "texte": texte,
                }
            )

        return pd.DataFrame(rows)

    except Exception as e:
        print(f" Erreur dans {fichier_xml} : {e}")
        return pd.DataFrame()


# ======== Fonction traitement d'un dossier contenant les XML =========
def traiter_dossier_compte_rendu_lxml(
    dossier_path: str, pattern: str = "*.xml"
) -> pd.DataFrame:
    """
    Traite tous les fichiers XML d'un dossier avec la fonction extraire_paragraphes_lxml().
    """
    # fichiers = glob.glob(os.path.join(dossier_path, pattern))
    # lecture des fichiers avec un sorted pour reproductibilité
    fichiers = sorted(glob.glob(os.path.join(dossier_path, pattern)))

    if not fichiers:
        print(f"Aucun fichier XML trouvé dans {dossier_path}")
        return pd.DataFrame()

    df_cumul = []
    total = len(fichiers)
    print(f"Traitement de {total} fichiers XML...\n")

    for i, fichier in enumerate(fichiers, 1):
        nom = os.path.basename(fichier)
        print(f"[{i}/{total}] {nom}...", end=" ")

        df_temp = extraire_paragraphes_lxml(fichier)
        if not df_temp.empty:
            print(f"{len(df_temp)} lignes")
            df_cumul.append(df_temp)
        else:
            print("Vide ou erreur")

    if df_cumul:
        df_extraction = pd.concat(df_cumul, ignore_index=True)
        print(f"\n Extraction terminée : {len(df_extraction)} lignes consolidées")
        return df_extraction
    else:
        return pd.DataFrame()


## 1.2 Extraction des données des XML et export CSV

In [13]:
# ==================================================================
# TRAITEMENT DES LÉGISLATURES SOUHAITÉES
# ==================================================================

# ========== Traitement des législatures ==========

# Traitement de la 16° législature
df_16 = traiter_dossier_compte_rendu_lxml("../data/raw/16-xml/compteRendu/")

# Traitement de la 15° législature
df_15 = traiter_dossier_compte_rendu_lxml("../data/raw/15-xml/compteRendu/")

# ========== Nettoyage fichiers doublons et congrès ==========

# UIDs à exclure (doublons / congrès)
"""
# nb tracabilité :
# NOTE: ici en manuel, mais pourrait imaginer une exclusion auto
# sur base de str.contains("Congrès du Parlement") dans session
# + ajouter deduplication sur base id_syceron + texte
# NOTE : la déduplication serait pas parfaite vs supr de fichier (voir dessous)
"""

uids_a_exclure = {
    "CRSANR5L16S2021O1N144",  # "faux" fichier en 16e (doublon de "CRSANR5L15S2021O1N144" de 2021)
    "CRSJOCGR5L15S2017E1N001",  # JO "Congrès du Parlement du 3 juillet 2017"
    "CRSANR5L15S2017O1N001",  # doublon AN JO "Congrès du Parlement du 3 juillet 2017"
    "CRSJOCGR5L15S2018E1N001",  # JO "Congrès du Parlement du 9 juillet 2018"
    "CRSCGR5L16S2024O1N001",  # CG "Congrès du Parlement du 4 mars 2024"
}

# Pour affichage (pas indispensable)
uids_trouvees_15 = uids_a_exclure & set(df_15["uid"])
uids_trouvees_16 = uids_a_exclure & set(df_16["uid"])

print(
    f"UIDs à exclure trouvés en df_15 : {len(uids_trouvees_15)}/{len(uids_a_exclure)}"
)
for uid in uids_trouvees_15:
    print(f"  - {uid}")

print(
    f"UIDs à exclure trouvés en df_16 : {len(uids_trouvees_16)}/{len(uids_a_exclure)}"
)
for uid in uids_trouvees_16:
    print(f"  - {uid}")

# puis suppression des lignes correspondantes
n15_avant, n16_avant = len(df_15), len(df_16)
df_15 = df_15[~df_15["uid"].isin(uids_a_exclure)]
df_16 = df_16[~df_16["uid"].isin(uids_a_exclure)]

print(f"Suppression UID ciblés - df_15 : {n15_avant - len(df_15)} ligne(s)")
print(f"Suppression UID ciblés - df_16 : {n16_avant - len(df_16)} ligne(s)")

# exports
df_16.to_csv("../data/interim/extract_16.csv", index=False, encoding="utf-8")
print(f"\n Export CSV df_16: ({df_16.shape[0]} lignes)")

df_15.to_csv("../data/interim/extract_15.csv", index=False, encoding="utf-8")
print(f"\n Export CSV df_15: ({df_15.shape[0]} lignes)")


Traitement de 605 fichiers XML...

[1/605] CRSANR5L16S2021O1N144.xml... 224 lignes
[2/605] CRSANR5L16S2022E1N001.xml... 584 lignes
[3/605] CRSANR5L16S2022E1N002.xml... 411 lignes
[4/605] CRSANR5L16S2022E1N003.xml... 374 lignes
[5/605] CRSANR5L16S2022E1N004.xml... 798 lignes
[6/605] CRSANR5L16S2022E1N005.xml... 577 lignes
[7/605] CRSANR5L16S2022E1N006.xml... 373 lignes
[8/605] CRSANR5L16S2022E1N007.xml... 646 lignes
[9/605] CRSANR5L16S2022E1N008.xml... 466 lignes
[10/605] CRSANR5L16S2022E1N009.xml... 835 lignes
[11/605] CRSANR5L16S2022E1N010.xml... 495 lignes
[12/605] CRSANR5L16S2022E1N011.xml... 831 lignes
[13/605] CRSANR5L16S2022E1N012.xml... 455 lignes
[14/605] CRSANR5L16S2022E1N013.xml... 569 lignes
[15/605] CRSANR5L16S2022E1N014.xml... 786 lignes
[16/605] CRSANR5L16S2022E1N015.xml... 1324 lignes
[17/605] CRSANR5L16S2022E1N016.xml... 680 lignes
[18/605] CRSANR5L16S2022E1N017.xml... 448 lignes
[19/605] CRSANR5L16S2022E1N018.xml... 810 lignes
[20/605] CRSANR5L16S2022E1N019.xml... 1015

## 1.3 Fusion des législatures

In [14]:
# ==================================================================
# FUSION DES LÉGISLATURES
# concaténation de df_15 et df_16 (ou lecture depuis CSV si nécessaire)
# ==================================================================

# # si déjà en mémoire : utiliser df_15, df_16 ; sinon :
# df_15 = pd.read_csv("../data/interim/extract_15.csv", encoding="utf-8")
# df_16 = pd.read_csv("../data/interim/extract_16.csv", encoding="utf-8")

# si besoin de vérifier et aligner les colonnes
# Mais overkill ici, on est propre normalement

# cols15 = set(df_15.columns)
# cols16 = set(df_16.columns)
# for c in sorted((cols15 | cols16) - cols15):
#     df_15[c] = pd.NA
# for c in sorted((cols15 | cols16) - cols16):
#     df_16[c] = pd.NA

# concat
df_concat = pd.concat([df_15, df_16], ignore_index=True, sort=False)

# ==================================================================
# DÉDUPLICATION
# ici utilisée en fallback après suppression ciblée de fichiers
# ==================================================================
"""
# NOTE : Pourrait supr la déduplication car ici = 0
# mais parce qu'on est allé identifier et supr les fichier doublons !
# On garde si évolution des données ou de choix de clé déduplication
# et pour repérage si ajout données
# NOTE : la deduplication est pas parfaite vs supr de fichier
# = des lignes qui passent le filtre car pas des vrais doublons
# (entête fichier, sans texte, etc. ?)
# par ex 7 lignes d'écart sur le fichier CRSANR5L16S2021O1N144
# donc autant virer les fichiers proprement quand on identifie
# et utiliser la déduplication en fallback
"""

# conservation de la clé texte pour éviter de supprimer certaines lignes
# qui ont même id_syceron mais texte différent (didascalies, etc.)
# ici le faire sans l'uid puisque l'enjeu c'est possiblement des fichiers pas nommés pareil !
# NOTE : enjeu identification = parfois mêmes "mauvais" uid pour les fichiers doublons
# ie : on peut pas juste aller regarder les noms de fichier avec l'iud, il correspond pas
dup_key = ["id_syceron", "texte"]

# mask pour les lignes qui seraient supprimées par drop_duplicates
mask_removed = df_concat.duplicated(subset=dup_key, keep="first")

if mask_removed.sum() == 0:
    print("Pas de doublons avec les clés choisies")
    print(f"concat {len(df_15)} + {len(df_16)} -> {len(df_concat)} lignes")
else:
    # lignes que l'on vire
    removed = df_concat[mask_removed].copy()
    # toutes les lignes impliquées dans un doublon, y compris celle qu'on garde
    mask_any = df_concat.duplicated(subset=dup_key, keep=False)
    # si besoin investigation :
    # groupes doublons = lignes dupliquées regroupées et triées
    # dupe_groups = df_concat[mask_any].sort_values(by=dup_key)
    # les survivants (si jamais on veut les voir, pas utilisé directement ici)
    # kept_in_groups = df_concat[~mask_removed & mask_any]

    print(f"Groupes dupliqués distincts  : {mask_removed.sum()}")
    print(f"Lignes supprimées prévues : {len(removed)}")
    # nb : si nb groupe = nb lignes c'est ok = paires doublons et pas triples etc.
    print("UIDs des lignes concernées (pas parfait pour retrouver fichier) :")
    for uid in sorted(removed["uid"].dropna().unique()):
        print(f"  - {uid}")

    df_concat_before = len(df_concat)
    df_concat = df_concat.drop_duplicates(subset=dup_key, keep="first")
    print(
        f"concat {len(df_15)} + {len(df_16)} -> {df_concat_before} lignes ; après déduplication {len(df_concat)} lignes"
    )

# export
df_concat.to_csv(
    "../data/interim/extract_15_16_concat.csv", index=False, encoding="utf-8"
)
print(f"\n Export CSV : ({df_concat.shape[0]} lignes)")

Pas de doublons avec les clés choisies
concat 791098 + 336740 -> 1127838 lignes

 Export CSV : (1127838 lignes)


# 1.4 regroupement des interventions interrompues

In [15]:
# ==================================================================
# REGROUPEMENT DES INTERVENTIONS INTERROMPUES
# Fusion des interventions d'un même orateur interrompues par des interruptions
# ==================================================================

# Repartir du df en mémoire
df = df_concat.copy()

# # ou le recharger depuis CSV (lui ou df souhaité)
# # Charger le df concaténé des deux législatures
# df = pd.read_csv(
#     "../data/interim/extract_15_16_concat.csv",
#     low_memory=False,
#     dtype={
#         "id_orateur": str  # éviter identification en float avant d'avoir ajouté le "PA"
#     },
# )

# print("Shape du df chargé : ", df.shape)

# Changer les missing values pour non_précisé (majoritaire) dans Code_parole
df["code_parole"] = df["code_parole"].fillna("non_précisé")

# # NOTE : après test ne semble pas dramatique de ne pas prendre en compte le
# # df["id_orateur"] = "PA" + df["id_orateur"] : seules 0 ou 2 lignes changent (si code parole ou pas)
# # les recodages "manuels" de PA repérés par ailleurs (voir autre notebook)
# # ne changent rien non plus ici
# # cf surtout des interruptions et ne change pas grand chose au regroup d'interventions
# # + quand erreur pas forcément de changement d'ID entre ou d'interv.

# # Mais par principe la trace si on veut garder :
# # Stabiliser le id_orateur pour être au format AN
# df["id_orateur"] = "PA" + df["id_orateur"]
# # Remplacer les valeurs manquantes de id_acteur par id_orateur quand disponible
# df["id_acteur_originel"] = df["id_acteur"]  # garder une trace
# df["id_acteur"] = df["id_acteur"].combine_first(df["id_orateur"])


"""
==========================
Regroupe les lignes de l'extraction CSV pour fusionner les interventions
d'un même orateur interrompues par des INTERRUPTION_1_10.

Sortie : un CSV entrelacé avec :
  - une ligne par groupe d'intervention fusionnée (texte concaténé)
  - les informations sur le nombre de fragments, d'interruptions reçues, etc.
  - les lignes INTERRUPTION conservées telles quelles, intercalées dans l'ordre
"""

# ---------------------------------------------------------------------------
# Paramètres
# ---------------------------------------------------------------------------

# Codes considérés comme interruptions (conservés tels quels dans la sortie)
CODES_INTERRUPTION = {"INTERRUPTION_1_10"}

# Colonnes invariantes dans un groupe (on garde la valeur de la 1ère ligne)
COLS_META = [
    "uid",
    "SeanceRef",
    "SessionRef",
    "dateSeance",
    "dateSeanceJour",
    "numSeanceJour",
    "numSeance",
    "typeAssemblee",
    "legislature",
    "session",
    "nomFichierJo",
    "presidentSeance",
    "point_titre",
    "point_type",
    "valeur_ptsodj",
    "ordinal_prise",
    "ordre_absolu_seance",
    "id_acteur",
    "id_mandat",
    "code_grammaire",
    "code_style",
    "code_parole",
    "id_syceron",
    "roledebat",
    "nom_orateur",
    "qualite_orateur",
    "id_orateur",
    "stime",
]

# ---------------------------------------------------------------------------
# Fonction principale
# ---------------------------------------------------------------------------


def regrouper(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prend un DataFrame trié par (uid, ordre_absolu_seance) et retourne
    un DataFrame entrelacé :
      - lignes d'intervention fusionnées (nb_fragments >= 1)
      - lignes d'interruption conservées telles quelles (nb_fragments = NaN)
    """
    cols_utiles = list(dict.fromkeys(COLS_META + ["texte"]))
    work = df[cols_utiles].copy()

    work["uid_norm"] = work["uid"].fillna("").astype(str)
    work["id_acteur_norm"] = work["id_acteur"].fillna("").astype(str)
    work["code_grammaire_norm"] = work["code_grammaire"].fillna("").astype(str)
    work["code_parole_norm"] = work["code_parole"].fillna("").astype(str)
    work["texte_norm"] = work["texte"].fillna("").astype(str)

    # # TODO : Ancien bug car faisait pas de conversion numérique et donc tri lexicographique sur les ordres de séance
    # TODO : tester avec id syceron pour ordre et voir ce qui casse ?
    # FIXME : repérérer cause de l'écart de lignes entre avec et sans tri avant regroupement
    # -> du a ordinal prise qui est absent pour le plus gros, mais reste un truc
    # avec le tri sans ordinal prise : Shape du df regroupé :  (981871, 33)
    # avec le tri et ordinal prise : Shape du df regroupé :  (960491, 33)
    # avec le tri seulement sur id_syceron : Shape du df regroupé :  (979438, 33)
    # Sans le tri : Shape du df regroupé :  (959660, 33)

    # NOTE : donc encore un écart, semble mieux de pas faire le tri, mais si le fait convertir en numérique
    # work['ordinal_prise_num'] = pd.to_numeric(work['ordinal_prise'], errors="coerce")
    # work['ordre_absolu_seance_num'] = pd.to_numeric(work['ordre_absolu_seance'], errors="coerce")
    # work = work.sort_values(['uid_norm', 'ordinal_prise_num', 'ordre_absolu_seance_num']).reset_index(drop=True)
    # et un autre test (PIRE) par ordre id_syceron pour voir :
    # work["id_syceron"] = pd.to_numeric(work["id_syceron"], errors="coerce")
    # n_nan = work["id_syceron"].isna().sum()
    # if n_nan > 0:
    #     raise ValueError(f"id_syceron manquant/non numérique sur {n_nan} ligne(s)")
    # work = work.sort_values(["id_syceron"]).reset_index(drop=True)

    resultats = []  # liste finale (interventions + interruptions)
    groupe = None  # groupe en cours d'accumulation
    buffer_interruptions = []  # interruptions entre deux fragments du même orateur

    def ligne_sortie_depuis_base(base_row: dict) -> dict:
        r = {col: base_row[col] for col in cols_utiles}
        r["nb_fragments"] = pd.NA
        r["nb_interruptions_recues"] = pd.NA
        r["a_ete_interrompu"] = pd.NA
        r["id_syceron_fragments"] = pd.NA
        # r["codes_gram_fragments"] = pd.NA # ie pour traçabilité si enlève condition
        # r["codes_parole_fragments"] = pd.NA # ie pour traçabilité si enlève condition
        # r["changement_code_grammaire"] = pd.NA # ie pour traçabilité si enlève condition
        # r["changement_code_parole"] = pd.NA # ie pour traçabilité si enlève condition
        return r

    def clore_groupe(g: dict) -> dict:
        """
        Finalise un groupe. Les interruptions du buffer seront émises APRÈS dans le flux.
        """
        row = g["premiere_ligne"].copy()
        row["texte"] = " ".join(
            g["textes"]
        )  # on prend les textes norm pour éviter les NaN
        row["nb_fragments"] = g["nb_fragments"]
        row["nb_interruptions_recues"] = g["nb_interruptions_recues"]
        row["a_ete_interrompu"] = g["nb_interruptions_recues"] > 0
        row["id_syceron_fragments"] = "|".join(g["codes_syceron"])
        # row["codes_gram_fragments"] = "|".join(g["codes_grammaire"]) # ie pour traçabilité si enlève condition
        # row["codes_parole_fragments"] = "|".join(g["codes_parole"]) # ie pour traçabilité si enlève condition
        # row["changement_code_grammaire"] = len(set(g["codes_grammaire"])) > 1 # ie pour traçabilité si enlève condition
        # row["changement_code_parole"] = len(set(g["codes_parole"])) > 1 # ie pour traçabilité si enlève condition
        return row

    records = work.to_dict("records")

    for row in records:
        cg = row["code_grammaire_norm"]
        cp = row["code_parole_norm"]
        acteur_str = row["id_acteur_norm"]
        uid_str = row["uid_norm"]
        syc = str(row["id_syceron"]) if pd.notna(row["id_syceron"]) else ""

        # --- Cas 1 : interruption ---
        if cg in CODES_INTERRUPTION:
            if groupe is not None:
                # L'interruption est dans le contexte d'un groupe ouvert :
                # on l'ajoute au buffer (elle sera émise si le même orateur reprend)
                buffer_interruptions.append(row)
                groupe["nb_interruptions_recues"] += 1
            else:
                # Interruption hors contexte (cas rare) : on l'émet directement
                resultats.append(ligne_sortie_depuis_base(row))
            continue

        # --- Cas 2 : intervention principale ---
        if (
            groupe is not None
            and buffer_interruptions  # on regroupe que si bien interrompu (et pas parle 2 fois de suite)
            and acteur_str != ""  # cf les nan convertis en ""
            and groupe["id_acteur"] == acteur_str
            and groupe["uid"] == uid_str
            and groupe["codes_grammaire"][-1] == cg
            and groupe["codes_parole"][-1] == cp
        ):
            # Même orateur, même séance, mêmes codes, avec interruption -> on fusionne
            groupe["textes"].append(row["texte_norm"])
            groupe["codes_grammaire"].append(cg)
            groupe["codes_parole"].append(cp)
            groupe["codes_syceron"].append(syc)
            groupe["nb_fragments"] += 1
        else:
            # Nouvel orateur ou nouvelle séance ou changement de codes
            if groupe is not None:
                # Clore le groupe précédent
                resultats.append(clore_groupe(groupe))
                # Et les interruptions en buffer suivent le groupe
                for irr in buffer_interruptions:
                    resultats.append(ligne_sortie_depuis_base(irr))
                buffer_interruptions = []

            groupe = {
                "uid": uid_str,
                "id_acteur": acteur_str,
                "premiere_ligne": {col: row[col] for col in cols_utiles},
                "textes": [row["texte_norm"]],
                "codes_grammaire": [cg],
                "codes_parole": [cp],
                "codes_syceron": [syc],
                "nb_fragments": 1,
                "nb_interruptions_recues": 0,
            }
    # Clore le dernier groupe
    if groupe is not None:
        resultats.append(clore_groupe(groupe))
        for irr in buffer_interruptions:
            resultats.append(ligne_sortie_depuis_base(irr))

    return pd.DataFrame(resultats)


df_interv_regroupe = regrouper(df)
print("Shape du df regroupé : ", df_interv_regroupe.shape)

df_interv_regroupe.to_csv("../data/interim/interventions_regroupees.csv", index=False)

Shape du df regroupé :  (959660, 33)


In [16]:
# TODO: check les ordres obsolu seances et si ils marchent ou pas (str vs int etc.)
# TODO : check pq de rugy marche pas ? CRSANR5L15S2019O1N196. Pb question au gouv comme ministre ? -> nope sans doute ordre de tri mauvais typage

# Tests bug regroup interventions

In [17]:
# =========================
# A) Profil des clés de tri
# =========================
w = df.copy()

w["ordinal_prise_num"] = pd.to_numeric(w["ordinal_prise"], errors="coerce")
w["ordre_absolu_seance_num"] = pd.to_numeric(w["ordre_absolu_seance"], errors="coerce")

print("Lignes totales:", len(w))
print("NaN ordinal_prise:", w["ordinal_prise_num"].isna().sum())
print("NaN ordre_absolu_seance:", w["ordre_absolu_seance_num"].isna().sum())

# UID les plus "sales"
uid_diag = (
    w.groupby("uid", dropna=False)
    .agg(
        n=("uid", "size"),
        n_nan_ord=("ordinal_prise_num", lambda s: s.isna().sum()),
        n_nan_abs=("ordre_absolu_seance_num", lambda s: s.isna().sum()),
        n_acteurs_vides=("id_acteur", lambda s: s.fillna("").eq("").sum()),
    )
    .sort_values(["n_nan_ord", "n_nan_abs", "n_acteurs_vides"], ascending=False)
)
print(uid_diag.head(20))

Lignes totales: 1127838
NaN ordinal_prise: 0
NaN ordre_absolu_seance: 0
                          n  n_nan_ord  n_nan_abs  n_acteurs_vides
uid                                                               
CRSANR5L15S2019O1N111  2079          0          0              415
CRSANR5L15S2021O1N122  1850          0          0              393
CRSANR5L15S2019O1N182  1883          0          0              392
CRSANR5L15S2020O1N108  1356          0          0              381
CRSANR5L15S2021E1N018  1640          0          0              297
CRSANR5L15S2018O1N276  1389          0          0              272
CRSANR5L15S2021O1N072  1440          0          0              272
CRSANR5L15S2018O1N101  1194          0          0              264
CRSANR5L16S2024O1N197   599          0          0              264
CRSANR5L15S2018O1N263  1401          0          0              262
CRSANR5L15S2019O1N194  1312          0          0              262
CRSANR5L15S2022O1N095   809          0          0        

In [18]:
# ===========================================
# B) Cas "interruption puis reprise" non fusionnés
# ===========================================
# On cherche le motif local:
# ligne i = intervention
# i+1 = INTERRUPTION_1_10
# i+2 = intervention même uid + même acteur
# mais un critère bloque la fusion

tmp = df.copy()
tmp["uid_norm"] = tmp["uid"].fillna("").astype(str)
tmp["id_acteur_norm"] = tmp["id_acteur"].fillna("").astype(str)
tmp["code_grammaire_norm"] = tmp["code_grammaire"].fillna("").astype(str)
tmp["code_parole_norm"] = tmp["code_parole"].fillna("non_précisé").astype(str)
tmp = tmp.reset_index(drop=True)

issues = []

for i in range(len(tmp) - 2):
    a = tmp.iloc[i]
    b = tmp.iloc[i + 1]
    c = tmp.iloc[i + 2]

    if b["code_grammaire_norm"] != "INTERRUPTION_1_10":
        continue
    if (
        a["code_grammaire_norm"] == "INTERRUPTION_1_10"
        or c["code_grammaire_norm"] == "INTERRUPTION_1_10"
    ):
        continue

    # reprise même orateur/séance attendue
    if (
        a["uid_norm"] == c["uid_norm"]
        and a["id_acteur_norm"] == c["id_acteur_norm"]
        and a["id_acteur_norm"] != ""
    ):
        blockers = []
        if a["code_grammaire_norm"] != c["code_grammaire_norm"]:
            blockers.append("code_grammaire_change")
        if a["code_parole_norm"] != c["code_parole_norm"]:
            blockers.append("code_parole_change")

        if blockers:
            issues.append(
                {
                    "uid": a["uid"],
                    "i": i,
                    "id_acteur": a["id_acteur"],
                    "nom_orateur": a.get("nom_orateur", None),
                    "ord_a": a.get("ordre_absolu_seance", None),
                    "ord_b": b.get("ordre_absolu_seance", None),
                    "ord_c": c.get("ordre_absolu_seance", None),
                    "id_syceron_a": a.get("id_syceron", None),
                    "id_syceron_b": b.get("id_syceron", None),
                    "id_syceron_c": c.get("id_syceron", None),
                    "blockers": "|".join(blockers),
                }
            )

issues_df = pd.DataFrame(issues)
print("Cas problématiques détectés:", len(issues_df))
display(issues_df.head(30))

Cas problématiques détectés: 2683


,uid,i,id_acteur,nom_orateur,ord_a,ord_b,ord_c,id_syceron_a,id_syceron_b,id_syceron_c,blockers
0,CRSANR5L15S2017E1N003,438,PA720430,M. Ugo Bernalicis,86,95,96,982693,982826,982702,code_grammaire_change|code_parole_change
1,CRSANR5L15S2017E1N003,529,PA720908,Mme Naïma Moutchou,250,259,260,982571,982793,982794,code_grammaire_change|code_parole_change
2,CRSANR5L15S2017E1N003,618,PA906,M. Gérard Collomb,492,494,495,983143,983146,983147,code_grammaire_change
3,CRSANR5L15S2017E1N005,1035,PA720422,M. Adrien Quatennens,232,235,236,984009,984015,984016,code_grammaire_change|code_parole_change
4,CRSANR5L15S2017E1N005,1089,PA610681,Mme Ericka Bareigts,324,341,342,984111,984129,984130,code_grammaire_change|code_parole_change
5,CRSANR5L15S2017E1N006,1317,PA721210,M. Alexis Corbière,119,123,124,984689,984817,984818,code_grammaire_change
6,CRSANR5L15S2017E1N007,1860,PA2150,M. Jean-Luc Mélenchon,307,310,311,986109,986111,986112,code_grammaire_change
7,CRSANR5L15S2017E1N007,1995,PA721418,M. Patrick Mignola,527,530,531,986817,986820,986821,code_grammaire_change
8,CRSANR5L15S2017E1N007,2032,PA721202,M. Éric Coquerel,606,608,609,986143,986175,986174,code_grammaire_change
9,CRSANR5L15S2017E1N007,2064,PA717169,Mme Muriel Pénicaud,671,683,684,986376,986302,986303,code_parole_change


In [19]:
# ======================================
# C) Comparaison avant/après tri (audit)
# ======================================
# Exécute regrouper 2 fois: sans tri, puis avec tri robuste, et compare.


def prepare_sorted_for_regroup(df_in):
    w = df_in.copy()
    w["uid_norm"] = w["uid"].fillna("").astype(str)

    w = w.reset_index(drop=False).rename(columns={"index": "_row_order"})
    w["ordinal_prise_num"] = pd.to_numeric(w["ordinal_prise"], errors="coerce")
    w["ordre_absolu_seance_num"] = pd.to_numeric(
        w["ordre_absolu_seance"], errors="coerce"
    )

    w = w.sort_values(
        by=["uid_norm", "ordinal_prise_num", "ordre_absolu_seance_num", "_row_order"],
        kind="mergesort",
        na_position="last",
    ).reset_index(drop=True)

    return w.drop(
        columns=["ordinal_prise_num", "ordre_absolu_seance_num", "_row_order"]
    )


# 1) sans tri
out_no_sort = regrouper(df.copy())

# 2) avec tri robuste
df_sorted = prepare_sorted_for_regroup(df.copy())
out_sort = regrouper(df_sorted)

print("Shape sans tri:", out_no_sort.shape)
print("Shape avec tri robuste:", out_sort.shape)

# Où ça change le plus (par uid)
a = out_no_sort.groupby("uid", dropna=False).size().rename("n_no_sort")
b = out_sort.groupby("uid", dropna=False).size().rename("n_sort")
delta = pd.concat([a, b], axis=1).fillna(0)
delta["delta"] = delta["n_sort"] - delta["n_no_sort"]
delta = delta.sort_values("delta", ascending=False)
display(delta.head(30))

Shape sans tri: (959660, 33)
Shape avec tri robuste: (960491, 33)


,n_no_sort,n_sort,delta
uid,,,
CRSANR5L15S2020O1N146,740,751,11
CRSANR5L15S2019O1N131,528,539,11
CRSANR5L15S2020O1N066,632,641,9
CRSANR5L15S2020O1N175,477,486,9
CRSANR5L15S2019O1N099,162,169,7
CRSANR5L15S2020O1N150,554,561,7
CRSANR5L15S2020O1N133,418,425,7
CRSANR5L15S2019O1N106,263,269,6
CRSANR5L15S2020O1N179,639,645,6


In [20]:
import pandas as pd


def build_triplets(df: pd.DataFrame) -> pd.DataFrame:
    t = df.copy().reset_index(drop=True)

    # Normalisations alignées avec regrouper
    t["uid_norm"] = t["uid"].fillna("").astype(str)
    t["id_acteur_norm"] = t["id_acteur"].fillna("").astype(str)
    t["code_grammaire_norm"] = t["code_grammaire"].fillna("").astype(str)
    t["code_parole_norm"] = t["code_parole"].fillna("non_précisé").astype(str)

    rows = []
    for i in range(len(t) - 2):
        a = t.iloc[i]
        b = t.iloc[i + 1]
        c = t.iloc[i + 2]

        # motif local: intervention -> interruption -> intervention
        if b["code_grammaire_norm"] != "INTERRUPTION_1_10":
            continue
        if a["code_grammaire_norm"] == "INTERRUPTION_1_10":
            continue
        if c["code_grammaire_norm"] == "INTERRUPTION_1_10":
            continue
        if a["uid_norm"] != c["uid_norm"]:
            continue

        same_actor = (a["id_acteur_norm"] != "") and (
            a["id_acteur_norm"] == c["id_acteur_norm"]
        )
        same_cg = a["code_grammaire_norm"] == c["code_grammaire_norm"]
        same_cp = a["code_parole_norm"] == c["code_parole_norm"]

        if same_actor and same_cg and same_cp:
            status = "fusion_attendue"
            reason = "ok_regles_fusion"
        else:
            status = "non_fusion"
            blockers = []
            if not same_actor:
                blockers.append("acteur_diff_ou_manquant")
            if not same_cg:
                blockers.append("code_grammaire_change")
            if not same_cp:
                blockers.append("code_parole_change")
            reason = "|".join(blockers)

        rows.append(
            {
                # clé de comparaison assez stable
                "triplet_key": f"{a.get('uid', '')}|{a.get('id_syceron', '')}|{b.get('id_syceron', '')}|{c.get('id_syceron', '')}",
                "uid": a.get("uid"),
                "nom_orateur_avant": a.get("nom_orateur"),
                "nom_orateur_reprise": c.get("nom_orateur"),
                "id_acteur_avant": a.get("id_acteur"),
                "id_acteur_reprise": c.get("id_acteur"),
                "ordre_avant": a.get("ordre_absolu_seance"),
                "ordre_interrupt": b.get("ordre_absolu_seance"),
                "ordre_reprise": c.get("ordre_absolu_seance"),
                "id_syceron_avant": a.get("id_syceron"),
                "id_syceron_interrupt": b.get("id_syceron"),
                "id_syceron_reprise": c.get("id_syceron"),
                "status": status,
                "reason": reason,
                "txt_avant": (a.get("texte") or "")[:180],
                "txt_interrupt": (b.get("texte") or "")[:180],
                "txt_reprise": (c.get("texte") or "")[:180],
            }
        )

    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.drop_duplicates(subset=["triplet_key"]).reset_index(drop=True)
    return out


def prepare_sorted_for_regroup(df_in: pd.DataFrame) -> pd.DataFrame:
    w = df_in.copy()
    w["uid_norm"] = w["uid"].fillna("").astype(str)
    w = w.reset_index(drop=False).rename(columns={"index": "_row_order"})
    w["ordinal_prise_num"] = pd.to_numeric(w["ordinal_prise"], errors="coerce")
    w["ordre_absolu_seance_num"] = pd.to_numeric(
        w["ordre_absolu_seance"], errors="coerce"
    )

    w = w.sort_values(
        by=["uid_norm", "ordinal_prise_num", "ordre_absolu_seance_num", "_row_order"],
        kind="mergesort",
        na_position="last",
    ).reset_index(drop=True)

    return w.drop(
        columns=["ordinal_prise_num", "ordre_absolu_seance_num", "_row_order"]
    )


# 1) Cas sans tri
cas_no_sort = build_triplets(df).rename(
    columns={"status": "status_no_sort", "reason": "reason_no_sort"}
)

# 2) Cas avec tri robuste
df_sorted = prepare_sorted_for_regroup(df)
cas_sort = build_triplets(df_sorted).rename(
    columns={"status": "status_sort", "reason": "reason_sort"}
)

# 3) Alignement et détection des changements
cols_common = [
    "triplet_key",
    "uid",
    "nom_orateur_avant",
    "nom_orateur_reprise",
    "id_acteur_avant",
    "id_acteur_reprise",
    "ordre_avant",
    "ordre_interrupt",
    "ordre_reprise",
    "id_syceron_avant",
    "id_syceron_interrupt",
    "id_syceron_reprise",
    "txt_avant",
    "txt_interrupt",
    "txt_reprise",
]

cmp = cas_no_sort[cols_common + ["status_no_sort", "reason_no_sort"]].merge(
    cas_sort[["triplet_key", "status_sort", "reason_sort"]],
    on="triplet_key",
    how="outer",
)

# classification lisible
cmp["status_no_sort"] = cmp["status_no_sort"].fillna("absent")
cmp["status_sort"] = cmp["status_sort"].fillna("absent")
cmp["reason_no_sort"] = cmp["reason_no_sort"].fillna("")
cmp["reason_sort"] = cmp["reason_sort"].fillna("")

changes = cmp[cmp["status_no_sort"] != cmp["status_sort"]].copy()

print("Triplets sans tri :", len(cas_no_sort))
print("Triplets avec tri :", len(cas_sort))
print("Triplets dont le statut change :", len(changes))

display(changes.sort_values(["uid", "ordre_avant"]).head(100))

# Export
changes.to_csv(
    "../data/interim/cas_statut_change_apres_tri.csv", index=False, encoding="utf-8"
)
print("Export:", "../data/interim/cas_statut_change_apres_tri.csv")

Triplets sans tri : 185981
Triplets avec tri : 185640
Triplets dont le statut change : 2817


,triplet_key,uid,nom_orateur_avant,nom_orateur_reprise,id_acteur_avant,id_acteur_reprise,ordre_avant,ordre_interrupt,ordre_reprise,id_syceron_avant,id_syceron_interrupt,id_syceron_reprise,txt_avant,txt_interrupt,txt_reprise,status_no_sort,reason_no_sort,status_sort,reason_sort
8871,CRSANR5L15S2018E1N036|1394690|1394697|1394683,CRSANR5L15S2018E1N036,Mme Catherine Fabre,M. le président,PA719570,PA1874,1143,1144,101,1394690,1394697,1394683,Il s’agit en effet de rétablir un alinéa suppr...,Ce n’était pas une erreur !,Quel est l’avis du Gouvernement ?,non_fusion,acteur_diff_ou_manquant|code_parole_change,absent,
8840,CRSANR5L15S2018E1N036|1393928|1393929|1393934,CRSANR5L15S2018E1N036,M. Pierre Dharréville,M. Pierre Dharréville,PA718926,PA718926,414,50,416,1393928,1393929,1393934,"Pour rassurer Mme Iborra, j’ai rencontré des c...",Nous aussi !,…vous pouvez cocher la case correspondante dan...,fusion_attendue,ok_regles_fusion,absent,
9039,CRSANR5L15S2018E1N037|1396559|1396671|1396562,CRSANR5L15S2018E1N037,M. Éric Ciotti,M. le président,PA330240,PA332747,21,962,963,1396559,1396671,1396562,Ces amendements visent à permettre à l’Office ...,Très bien !,Quel est l’avis de la commission ?,non_fusion,acteur_diff_ou_manquant|code_grammaire_change|...,absent,
9054,CRSANR5L15S2018E1N038|1396071|1396527|1396434,CRSANR5L15S2018E1N038,M. Jean-Paul Dufrègne,M. Jean-Paul Dufrègne,PA718720,PA718720,208,213,3,1396071,1396527,1396434,…simplement parce qu’elle demandait des argume...,"Merci, monsieur Dufrègne…","Je termine, monsieur le président. Troisièmeme...",non_fusion,code_grammaire_change|code_parole_change,absent,
9100,CRSANR5L15S2018E1N038|1397021|1397023|1397405,CRSANR5L15S2018E1N038,Mme Sonia Krimi,M. le président,PA720202,PA332747,26,28,862,1397021,1397023,1397405,Je répondrai aux propos de mon collègue Floren...,Elle finira dans l’opposition plus vite que pr...,Je vais maintenant mettre aux voix les amendem...,non_fusion,acteur_diff_ou_manquant|code_grammaire_change|...,absent,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36119,CRSANR5L15S2019E1N002|1793317|1793318|1793320,CRSANR5L15S2019E1N002,None,Mme la présidente,None,PA605991,510,511,1,1793317,1793318,1793320,(Les amendements identiques nos\n ...,"À deux heures moins le quart du matin, enfin u...","La parole est à Mme Michèle Victory, pour sout...",non_fusion,acteur_diff_ou_manquant|code_grammaire_change,absent,
36291,CRSANR5L15S2019E1N004|1796083|1796130|1796084,CRSANR5L15S2019E1N004,Mme Nicole Belloubet,None,PA702191,None,12,413,414,1796083,1796130,1796084,Même avis.,Heureusement !,(L’amendement no\n \n ...,non_fusion,acteur_diff_ou_manquant|code_grammaire_change|...,absent,
36349,CRSANR5L15S2019E1N005|1797759|1797767|1797768,CRSANR5L15S2019E1N005,Mme Véronique Louwagie,Mme Véronique Louwagie,PA608016,PA608016,4,178,179,1797759,1797767,1797768,"Néanmoins, les deux articles ont un point comm...",Très juste !,Nous souhaitons de tout cœur qu’une solution s...,fusion_attendue,ok_regles_fusion,absent,
36364,CRSANR5L15S2019E1N006|1797068|1797069|1797070,CRSANR5L15S2019E1N006,Mme Caroline Abadie,None,PA719866,None,290,44,45,1797068,1797069,1797070,Le notifiant ne doit être tenu de décliner ses...,Mais c’est la plateforme qui décide !,(L’amendement no\n \n 3...,non_fusion,acteur_diff_ou_manquant|code_grammaire_change,absent,


Export: ../data/interim/cas_statut_change_apres_tri.csv


In [21]:
changes[changes["nom_orateur_avant"].fillna("").str.contains("rugy", case=False)][
    [
        "uid",
        "status_no_sort",
        "status_sort",
        "reason_no_sort",
        "reason_sort",
        "ordre_avant",
        "ordre_interrupt",
        "ordre_reprise",
        "id_syceron_avant",
        "id_syceron_interrupt",
        "id_syceron_reprise",
    ]
].head(100)

,uid,status_no_sort,status_sort,reason_no_sort,reason_sort,ordre_avant,ordre_interrupt,ordre_reprise,id_syceron_avant,id_syceron_interrupt,id_syceron_reprise
44065,CRSANR5L15S2019O1N062,fusion_attendue,absent,ok_regles_fusion,,167,169,30,1512204,1512405,1512206
49220,CRSANR5L15S2019O1N124,fusion_attendue,absent,ok_regles_fusion,,258,260,18,1590342,1590314,1590316
50440,CRSANR5L15S2019O1N141,fusion_attendue,absent,ok_regles_fusion,,3,270,271,1608961,1608979,1608980
54844,CRSANR5L15S2019O1N202,fusion_attendue,absent,ok_regles_fusion,,598,710,711,1684487,1684488,1684489
55122,CRSANR5L15S2019O1N204,fusion_attendue,absent,ok_regles_fusion,,21,272,273,1686876,1686953,1686952
56901,CRSANR5L15S2019O1N224,fusion_attendue,absent,ok_regles_fusion,,76,45,46,1708604,1708612,1708613
60930,CRSANR5L15S2019O1N290,fusion_attendue,absent,ok_regles_fusion,,19,22,376,1782340,1782343,1782344
60931,CRSANR5L15S2019O1N290,fusion_attendue,absent,ok_regles_fusion,,376,31,385,1782344,1782352,1782353
60987,CRSANR5L15S2019O1N290,fusion_attendue,absent,ok_regles_fusion,,5,6,580,1783128,1783129,1783132
61138,CRSANR5L15S2019O1N293,non_fusion,absent,acteur_diff_ou_manquant|code_grammaire_change|...,,285,286,22,1785742,1785887,1785743


In [22]:
# a tester :

import pandas as pd

# 1) ID stable pour tracer les mêmes lignes dans tous les scénarios
base = df.copy().reset_index(drop=True)
base["_row_id"] = base.index
base["uid_norm"] = base["uid"].fillna("").astype(str)

# 2) variantes de tri
v_no = base.copy()

v_ord = base.copy()
v_ord["ordinal_prise_num"] = pd.to_numeric(v_ord["ordinal_prise"], errors="coerce")
v_ord["ordre_abs_num"] = pd.to_numeric(v_ord["ordre_absolu_seance"], errors="coerce")
v_ord = v_ord.sort_values(
    ["uid_norm", "ordinal_prise_num", "ordre_abs_num", "_row_id"],
    kind="mergesort",
    na_position="last",
).reset_index(drop=True)

v_syc = base.copy()
v_syc["id_syceron_num"] = pd.to_numeric(v_syc["id_syceron"], errors="coerce")
n_nan = v_syc["id_syceron_num"].isna().sum()
if n_nan:
    raise ValueError(f"id_syceron manquant/non numérique: {n_nan}")
v_syc = v_syc.sort_values(["id_syceron_num", "_row_id"], kind="mergesort").reset_index(drop=True)

# 3) où l'ordre diverge, par uid
def first_divergence_by_uid(a, b):
    out = []
    au = a.groupby("uid_norm")["_row_id"].apply(list)
    bu = b.groupby("uid_norm")["_row_id"].apply(list)
    for uid in sorted(set(au.index).intersection(bu.index)):
        la, lb = au[uid], bu[uid]
        m = min(len(la), len(lb))
        k = next((i for i in range(m) if la[i] != lb[i]), None)
        if k is not None or len(la) != len(lb):
            out.append({
                "uid": uid,
                "len_a": len(la),
                "len_b": len(lb),
                "first_diff_pos": -1 if k is None else k,
                "row_a_at_diff": None if k is None else la[k],
                "row_b_at_diff": None if k is None else lb[k],
            })
    return pd.DataFrame(out).sort_values(["first_diff_pos", "uid"])

diff_no_vs_ord = first_divergence_by_uid(v_no, v_ord)
diff_no_vs_syc = first_divergence_by_uid(v_no, v_syc)

print("UID avec divergence ordre (no vs ord):", len(diff_no_vs_ord))
print("UID avec divergence ordre (no vs syc):", len(diff_no_vs_syc))
display(diff_no_vs_ord.head(20))
display(diff_no_vs_syc.head(20))

UID avec divergence ordre (no vs ord): 556
UID avec divergence ordre (no vs syc): 2158


,uid,len_a,len_b,first_diff_pos,row_a_at_diff,row_b_at_diff
409,CRSANR5L15S2020O1N079,567,567,8,456203,456204
513,CRSANR5L15S2020O1N198,305,305,9,518545,518546
548,CRSANR5L15S2020O1N241,485,485,10,535799,535803
31,CRSANR5L15S2019E1N002,820,820,11,214168,214171
121,CRSANR5L15S2019O1N058,349,349,11,263381,263382
186,CRSANR5L15S2019O1N131,660,660,11,305159,305166
147,CRSANR5L15S2019O1N087,256,256,12,281605,281606
157,CRSANR5L15S2019O1N099,206,206,13,287607,287609
226,CRSANR5L15S2019O1N176,524,524,14,325497,325498
549,CRSANR5L15S2020O1N242,152,152,14,536288,536289


,uid,len_a,len_b,first_diff_pos,row_a_at_diff,row_b_at_diff
1,CRSANR5L15S2017E1N002,153,153,0,261,284
3,CRSANR5L15S2017E1N004,209,209,0,705,905
7,CRSANR5L15S2017E1N008,451,451,0,2306,2372
21,CRSANR5L15S2017E1N022,580,580,0,10298,10544
24,CRSANR5L15S2017E1N025,694,694,0,12340,12628
35,CRSANR5L15S2017E2N003,844,844,0,19340,19438
36,CRSANR5L15S2017E2N004,708,708,0,20184,20539
38,CRSANR5L15S2017E2N006,626,626,0,21641,21977
41,CRSANR5L15S2017O1N124,26,26,0,23820,23835
50,CRSANR5L15S2018E1N008,259,259,0,27708,27899
